In [181]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [182]:
jobs = pd.read_csv("../data/processed/jobs_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", jobs.shape)

jobs.head()

Dataset loaded successfully!
Dataset shape: (1020, 13)


,id,title,company,location,salary_range,workplace_type,employment_type,description,clean_description,extracted_skills,all_skills,technical_skills,professional_skills
0,1,Machine Learning Engineering Internship - Summer 2026,Viant Technology,"Irvine, CA",$40/hr - $50/hr,Hybrid,Internship,What You’ll Do\n\n\n\nViant is seeking a Machine Learning Engineering Intern to support the deve...,what you’ll do viant is seeking a machine learning engineering intern to support the development...,"{'programming_languages': ['python', 'sql'], 'machine_learning_ai': ['machine learning', 'artifi...","['python', 'sql', 'machine learning', 'artificial intelligence', 'data science', 'data pipelines...","['python', 'sql', 'machine learning', 'artificial intelligence', 'data science', 'data pipelines...",[]
1,2,NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,NVIDIA,"Santa Clara, CA",NaN,On-site,Full-time,"By submitting your resume, you’re expressing interest in one of our 2026 Deep Learning Computer ...","by submitting your resume, you’re expressing interest in one of our 2026 deep learning computer ...","{'programming_languages': ['python', 'c++'], 'machine_learning_ai': ['deep learning', 'neural ne...","['python', 'c++', 'deep learning', 'neural networks', 'tensorflow', 'pytorch', 'keras', 'compute...","['python', 'c++', 'deep learning', 'neural networks', 'tensorflow', 'pytorch', 'keras', 'compute...",[]
2,3,Summer 2026 Intern - Undergrad User Research - Slack,Salesforce,"San Francisco, CA",$53/hr - $55/hr,Hybrid,Full-time,"To get the best candidate experience, please consider applying for a maximum of 3 roles within 1...","to get the best candidate experience, please consider applying for a maximum of 3 roles within 1...","{'programming_languages': [], 'machine_learning_ai': ['artificial intelligence'], 'data_science'...","['artificial intelligence', 'statistics', 'communication', 'leadership', 'collaboration', 'resea...","['artificial intelligence', 'statistics']","['communication', 'leadership', 'collaboration', 'research']"
3,4,Summer 2026 Engineering Intern,Truveta,Greater Seattle Area,$20/hr,Hybrid,Full-time,Truveta is the world’s first health provider led data platform with a vision of Saving Lives wit...,truveta is the world’s first health provider led data platform with a vision of saving lives wit...,"{'programming_languages': ['python', 'java', 'c#'], 'machine_learning_ai': [], 'data_science': [...","['python', 'java', 'c#', 'big data', 'communication', 'leadership', 'time management']","['python', 'java', 'c#', 'big data']","['communication', 'leadership', 'time management']"
4,5,Applied AI Engineer (Internship),Fortinet,"Santa Clara, CA",NaN,On-site,Full-time,Job Description\n\nWe’re a cybersecurity company building a next-generation AI-driven operations...,job description we’re a cybersecurity company building a next-generation ai-driven operations pl...,"{'programming_languages': ['python', 'go'], 'machine_learning_ai': ['generative ai'], 'data_scie...","['python', 'go', 'generative ai', 'cybersecurity', 'communication', 'collaboration']","['python', 'go', 'generative ai', 'cybersecurity']","['communication', 'collaboration']"


In [183]:
print("Dataset shape:", jobs.shape)

print("\nDataset columns:")
print(jobs.columns.tolist())

print("\nMissing values:")
print(jobs.isnull().sum())

Dataset shape: (1020, 13)

Dataset columns:
['id', 'title', 'company', 'location', 'salary_range', 'workplace_type', 'employment_type', 'description', 'clean_description', 'extracted_skills', 'all_skills', 'technical_skills', 'professional_skills']

Missing values:
id                       0
title                    0
company                  0
location                 0
salary_range           400
workplace_type           0
employment_type          0
description              0
clean_description        0
extracted_skills         0
all_skills               0
technical_skills         0
professional_skills      0
dtype: int64


In [184]:
# Replace missing descriptions with empty strings
jobs['clean_description'] = jobs['clean_description'].fillna('')

# Check the first three descriptions
jobs[['title', 'clean_description']].head(3)

,title,clean_description
0,Machine Learning Engineering Internship - Summer 2026,what you’ll do viant is seeking a machine learning engineering intern to support the development...
1,NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,"by submitting your resume, you’re expressing interest in one of our 2026 deep learning computer ..."
2,Summer 2026 Intern - Undergrad User Research - Slack,"to get the best candidate experience, please consider applying for a maximum of 3 roles within 1..."


**Apply TF-IDF vectorization**

In [185]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(jobs['clean_description'])

print("TF-IDF vectorization completed!")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF vectorization completed!
TF-IDF matrix shape: (1020, 5000)


**Calculate cosine similarity**

In [186]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity calculated successfully!")
print("Similarity matrix shape:", cosine_sim.shape)

Cosine similarity calculated successfully!
Similarity matrix shape: (1020, 1020)


**Examine job similarities**

In [187]:
similarity_df = pd.DataFrame(
    cosine_sim[:3, :3],
    index=jobs['title'][:3],
    columns=jobs['title'][:3]
)

similarity_df

title,Machine Learning Engineering Internship - Summer 2026,NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,Summer 2026 Intern - Undergrad User Research - Slack
title,,,
Machine Learning Engineering Internship - Summer 2026,1.000000,0.132228,0.126071
NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,0.132228,1.000000,0.101932
Summer 2026 Intern - Undergrad User Research - Slack,0.126071,0.101932,1.000000


**Recommendation function**

In [188]:
def recommend_jobs(job_title, jobs, similarity_matrix, top_n=5, company=None):

    matching_jobs = jobs[
        jobs['title'].str.lower() == job_title.lower()
    ]

    if company is not None:
        matching_jobs = matching_jobs[
            matching_jobs['company'].str.lower() == company.lower()
        ]

    if matching_jobs.empty:
        return "Job not found!"

    job_index = matching_jobs.index[0]

    selected_title = jobs.loc[job_index, 'title'].strip().lower()
    selected_company = jobs.loc[job_index, 'company'].strip().lower()
    selected_id = jobs.loc[job_index, 'id']

    similarity_scores = list(enumerate(similarity_matrix[job_index]))

    # Sort by similarity
    similarity_scores.sort(key=lambda x: x[1], reverse=True)

    top_jobs = []

    for index, score in similarity_scores:

        candidate_id = jobs.loc[index, 'id']
        candidate_title = jobs.loc[index, 'title'].strip().lower()
        candidate_company = jobs.loc[index, 'company'].strip().lower()

        # Exclude the exact selected job
        if candidate_id == selected_id:
            continue

        # Exclude identical titles from the same company
        if (candidate_title == selected_title and
                candidate_company == selected_company):
            continue

        top_jobs.append((index, score))

        if len(top_jobs) == top_n:
            break

    recommended_indices = [index for index, score in top_jobs]

    recommendations = jobs.iloc[recommended_indices][
        ['title', 'company', 'location', 'employment_type']
    ].copy()

    recommendations['similarity_score'] = [
        score for index, score in top_jobs
    ]

    return recommendations

In [189]:
recommendations = recommend_jobs(
    "Machine Learning Engineering Internship - Summer 2026",
    jobs,
    cosine_sim,
    top_n=5
)

recommendations

,title,company,location,employment_type,similarity_score
776,Intern - Software Engineering,Rakuten Advertising,"Salt Lake City, UT",Full-time,0.324019
305,ML-AI Engineering Intern,Tech Economy,"San Francisco, CA",Full-time,0.297595
287,"Intern, AI Engineering-2",Cast & Crew,"Burbank, CA",Full-time,0.251068
383,"Intern, AI Engineer",Cast & Crew,United States,Full-time,0.251068
603,Software Engineer Intern 2026,Creatify AI,"Mountain View, CA",Full-time,0.247391


In [190]:
research_recommendations = recommend_jobs(
    "Summer 2026 Intern - Undergrad User Research - Slack",
    jobs,
    cosine_sim,
    top_n=5
)

research_recommendations

,title,company,location,employment_type,similarity_score
5,Summer 2026 Intern - Undergrad User Research - Slack,Slack,"San Francisco, CA",Full-time,1.000000
78,Summer 2026 Intern - Applied AI Strategy & Research,Salesforce,"San Francisco, CA",Full-time,0.782701
115,User Researcher Intern,Lensa,"San Francisco, CA",Internship,0.279971
72,"Technical Sourcer, Research",OpenAI,"San Francisco, CA",Full-time,0.255697
24,"UX Researcher, AI",GitLab,United States,Full-time,0.248345


In [191]:
jobs[['title', 'technical_skills']].head(5)

,title,technical_skills
0,Machine Learning Engineering Internship - Summer 2026,"['python', 'sql', 'machine learning', 'artificial intelligence', 'data science', 'data pipelines..."
1,NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,"['python', 'c++', 'deep learning', 'neural networks', 'tensorflow', 'pytorch', 'keras', 'compute..."
2,Summer 2026 Intern - Undergrad User Research - Slack,"['artificial intelligence', 'statistics']"
3,Summer 2026 Engineering Intern,"['python', 'java', 'c#', 'big data']"
4,Applied AI Engineer (Internship),"['python', 'go', 'generative ai', 'cybersecurity']"


**Improve model using technical skills**

In [192]:
import ast

jobs['technical_skills'] = jobs['technical_skills'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

jobs['technical_skills'].head()

0    [python, sql, machine learning, artificial intelligence, data science, data pipelines, tensorflo...
1    [python, c++, deep learning, neural networks, tensorflow, pytorch, keras, computer architecture,...
2                                                                  [artificial intelligence, statistics]
3                                                                           [python, java, c#, big data]
4                                                             [python, go, generative ai, cybersecurity]
Name: technical_skills, dtype: object

In [193]:
# Convert skills into text
jobs['skills_text'] = jobs['technical_skills'].apply(
    lambda skills: ' '.join(skills)
)

jobs[['title', 'skills_text']].head()

,title,skills_text
0,Machine Learning Engineering Internship - Summer 2026,python sql machine learning artificial intelligence data science data pipelines tensorflow pytor...
1,NVIDIA 2026 Internships: Deep Learning Computer Architecture - US,python c++ deep learning neural networks tensorflow pytorch keras computer architecture gpu comp...
2,Summer 2026 Intern - Undergrad User Research - Slack,artificial intelligence statistics
3,Summer 2026 Engineering Intern,python java c# big data
4,Applied AI Engineer (Internship),python go generative ai cybersecurity


**Create the skill-based TF-IDF matrix**

In [194]:
skill_tfidf = TfidfVectorizer()

skill_matrix = skill_tfidf.fit_transform(jobs['skills_text'])

print("Skill TF-IDF matrix shape:", skill_matrix.shape)

Skill TF-IDF matrix shape: (1020, 87)


*Calculate skill similarity*

In [195]:
skill_similarity = cosine_similarity(skill_matrix, skill_matrix)

print("Skill similarity matrix shape:", skill_similarity.shape)

Skill similarity matrix shape: (1020, 1020)


*Test the results*

In [196]:
skill_recommendations = recommend_jobs(
    "Machine Learning Engineering Internship - Summer 2026",
    jobs,
    skill_similarity,
    top_n=5
)

skill_recommendations

,title,company,location,employment_type,similarity_score
62,Machine Learning Engineer,RADAR,San Diego Metropolitan Area,Full-time,0.633509
231,"Internship, Machine Learning Engineer, Data Engineering & Analytics (Summer 2026)",Tesla,"Fremont, CA",Internship,0.596559
46,Python Developer (Junior - Mid Level) - Fully Remote,Bravko,United States,Full-time,0.588901
755,Applied AI Engineer,Valence,"New York, NY",Full-time,0.579417
120,Generative AI Platform Engineer,Visionaire Partners,Atlanta Metropolitan Area,Contract,0.576326


**Build the combined content-based model**

In [197]:
# Assign weights to each similarity model
description_weight = 0.3
skill_weight = 0.7

# Combine the two similarity matrices
combined_similarity = (
    description_weight * cosine_sim +
    skill_weight * skill_similarity
)

print("Combined similarity matrix shape:", combined_similarity.shape)

Combined similarity matrix shape: (1020, 1020)


In [198]:
hybrid_recommendations = recommend_jobs(
    "Machine Learning Engineering Internship - Summer 2026",
    jobs,
    combined_similarity,
    top_n=5
)

hybrid_recommendations

,title,company,location,employment_type,similarity_score
62,Machine Learning Engineer,RADAR,San Diego Metropolitan Area,Full-time,0.513003
755,Applied AI Engineer,Valence,"New York, NY",Full-time,0.474762
231,"Internship, Machine Learning Engineer, Data Engineering & Analytics (Summer 2026)",Tesla,"Fremont, CA",Internship,0.473904
288,"Intern, AI Engineering",PowerSchool,"Dallas, TX",Full-time,0.465447
680,Data Engineer Intern,CopperPoint Insurance Companies,"Seattle, WA",Part-time,0.461060


In [199]:
research_hybrid_recommendations = recommend_jobs(
    "Summer 2026 Intern - Undergrad User Research - Slack",
    jobs,
    combined_similarity,
    top_n=5
)

research_hybrid_recommendations

,title,company,location,employment_type,similarity_score
5,Summer 2026 Intern - Undergrad User Research - Slack,Slack,"San Francisco, CA",Full-time,1.000000
78,Summer 2026 Intern - Applied AI Strategy & Research,Salesforce,"San Francisco, CA",Full-time,0.578124
85,Intern,UBS,"Denver, CO",Full-time,0.534745
92,Intern,UBS,"Denver, CO",Part-time,0.534596
91,Business Development Analyst,Etsy,"Brooklyn, NY",Full-time,0.534525


In [200]:
import os

os.makedirs("../models", exist_ok=True)

np.save(
    "../models/description_similarity.npy",
    cosine_sim
)

np.save(
    "../models/skill_similarity.npy",
    skill_similarity
)

np.save(
    "../models/combined_similarity.npy",
    combined_similarity
)

print("Content-based similarity models saved successfully!")

Content-based similarity models saved successfully!
